# M5 Walmart Demand Forecasting — Phase 1: Exploratory Data Analysis

**Store scope:** CA_1  
**Granularity:** Daily aggregated sales (all items combined)  
**Date range:** Jan 2011 – May 2016  


---
## 1. Business Context

Walmart operates thousands of stores across the US, stocking hundreds of thousands of SKUs. Accurate demand forecasting is critical for:

- **Inventory optimisation** — reducing overstock holding costs (~20–30% of item value/year)
- **Avoiding stockouts** — lost sales and customer churn
- **Supply chain scheduling** — vendor lead times, distribution centre routing
- **Promotional planning** — SNAP days, holidays, event-driven demand spikes

**Goal of this notebook:** Build a deep understanding of the CA_1 sales signal — its trend, seasonality structure, anomalies, and stationarity — before selecting and tuning forecasting models.


---
## 2. Data Loading & Scoping (CA_1 Store Only)


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from src.data_loader import load_ca1_daily

df = load_ca1_daily()
print(df.shape)
df.head()

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
df["sales"].describe()

---
## 3. Time Series Overview


In [ ]:
from src.eda import plot_sales_over_time

fig = plot_sales_over_time(df, rolling_window=28)
plt.show()

---
## 4. Trend Analysis


In [ ]:
from src.eda import plot_yearly_trend

fig = plot_yearly_trend(df)
plt.show()

In [ ]:
# STL decomposition — requires a complete, regular time series
from statsmodels.tsa.seasonal import STL

sales_series = df.set_index("date")["sales"]
stl = STL(sales_series, period=7, robust=True)
result = stl.fit()
result.plot()
plt.tight_layout()
plt.show()

---
## 5. Seasonality Analysis (Weekly / Monthly / Yearly)


In [ ]:
from src.eda import plot_weekly_seasonality, plot_monthly_seasonality

fig = plot_weekly_seasonality(df)
plt.show()

fig = plot_monthly_seasonality(df)
plt.show()

In [ ]:
# SNAP days impact
snap_vs_normal = df.groupby("snap_day")["sales"].mean()
print("Average sales — SNAP vs non-SNAP days:")
print(snap_vs_normal)

In [ ]:
# Holiday impact
holiday_vs_normal = df.groupby("is_holiday")["sales"].mean()
print("Average sales — Holiday vs non-Holiday days:")
print(holiday_vs_normal)

---
## 6. Anomaly Detection & Business Explanation


In [ ]:
from src.eda import detect_anomalies_zscore, plot_anomalies

anomalies = detect_anomalies_zscore(df, threshold=3.0)
print(f"Anomaly days detected: {len(anomalies)}")
anomalies[["date", "sales", "z_score", "is_holiday", "snap_day"]]

In [ ]:
fig = plot_anomalies(df, anomalies)
plt.show()

---
## 7. Stationarity Test (ADF)


In [ ]:
from src.eda import run_adf_test

print("=== ADF Test — Raw Sales ===")
adf_raw = run_adf_test(df["sales"])

In [ ]:
# If not stationary, apply first differencing
print("=== ADF Test — First-Differenced Sales ===")
adf_diff = run_adf_test(df["sales"].diff().dropna())

---
## 8. Key EDA Findings Summary

| Finding | Detail |
|---|---|
| Trend | *(fill in after running cells)* |
| Weekly seasonality | *(e.g., weekend uplift of X%)* |
| Monthly seasonality | *(e.g., December peak)* |
| SNAP day effect | *(e.g., +Y% on SNAP days)* |
| Holiday effect | *(e.g., Christmas dip / pre-holiday spike)* |
| Anomalies | *(N anomalies detected; key dates)* |
| Stationarity | *(ADF result — stationary / non-stationary; d=?)* |
| Recommended model features | *(e.g., day_of_week, snap_day, lag_7, lag_28)* |

**Next step:** → `notebooks/02_models.ipynb` — train SARIMA, Prophet, and LSTM and compare forecast accuracy.
